# Formation efficiency per binary -- legacy COMPAS models

This notebook computes the **formation efficiency per binary** for double
compact objects (DCOs) in *legacy* COMPAS output files, i.e. the model suite
used in [Broekgaarden et al. 2022](https://arxiv.org/abs/2103.02608) and
similar runs (a single `COMPASOutput.h5` file per model, with
`doubleCompactObjects` / `systems` / `supernovae` / ... groups).

The formation efficiency of a binary is

```
formation_efficiency = weight / total_mass_evolved_at_that_binary's_metallicity   [Msun^-1]
```

i.e. the number of such binaries formed per solar mass of star formation.

All the underlying logic now lives in
[`fe_legacy_COMPAS_functions.py`](./fe_legacy_COMPAS_functions.py) (in this
same folder). Compared to the old `fe_legacy_compas_models.ipynb` +
`legacy_compas_scripts/`:

- No `ClassCOMPAS.COMPASData` object -- functions read only the handful of
  HDF5 columns they actually need with `h5py`, instead of loading dozens of
  unused 17M+ element arrays into memory.
- No `alphabetDirDict` / BPS-model-letter indirection -- you just pass the
  path to a `COMPASOutput.h5` file directly.
- The output is a per-binary `pandas.DataFrame` (seed, metallicity, weight,
  formation efficiency, + optional extra columns), which is easy to save,
  filter, and combine with other (non-legacy) formation-efficiency /
  cosmic-integration pipelines later on.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

from fe_legacy_COMPAS_functions import calculate_formation_efficiency_per_binary

## Settings

Point `path` at a legacy `COMPASOutput.h5` file, and choose the DCO type and
the COMPAS sampling bounds (these come from the `pythonSubmit` used to run
that model, i.e. `--initial-mass-min` / `--initial-mass-max` /
`--mass-ratio-...` -> binary fraction).

In [ ]:
# path to a legacy COMPAS DCO output file (e.g. one of the Broekgaarden+22 models)
path = '/Volumes/SimonsFoundation/DataDCO/fiducial/COMPASOutput.h5'

dco_type = 'BHNS'         # one of 'BBH', 'BNS', 'BHNS'

# COMPAS primary-mass sampling bounds [Msun] and assumed binary fraction
# (these are the values used for all Broekgaarden+22 models)
m1_min = 5.0
m1_max = 150.0
binary_fraction = 1.0

## Compute the formation efficiency per binary

`calculate_formation_efficiency_per_binary` does everything in one call:

1. Works out the metallicity grid simulated by COMPAS and the total stellar
   mass formed at each metallicity (`get_total_mass_evolved_per_metallicity`,
   using a Kroupa-IMF Monte-Carlo normalisation -- pass `random_seed` for
   reproducibility).
2. Selects the requested DCO type, applying the standard
   merges-within-a-Hubble-time / pessimistic-CE / no-immediate-RLOF-after-CE
   cuts (`get_dco_mask`; these can be relaxed via keyword arguments).
3. Returns one row per surviving binary with its formation efficiency.

In [ ]:
binaries, metallicity_grid, total_mass_evolved_per_Z = calculate_formation_efficiency_per_binary(
    path,
    dco_type=dco_type,
    m1_min=m1_min,
    m1_max=m1_max,
    binary_fraction=binary_fraction,
    random_seed=42,  # for a reproducible IMF-sampling normalisation
)

print(f'{len(binaries)} {dco_type} binaries selected')
print(f'metallicity grid: {len(metallicity_grid)} points, '
      f'{metallicity_grid.min():.2e} - {metallicity_grid.max():.2e}')

## Inspect the per-binary table

`binaries` has one row per selected DCO. `formation_efficiency` is in units
of Msun^-1 (number of such binaries formed per solar mass of star formation
at that binary's metallicity).

In [ ]:
binaries.head()

In [ ]:
binaries['formation_efficiency'].describe()

## Formation efficiency vs. metallicity

Summing the per-binary formation efficiencies in each metallicity bin gives
the total formation efficiency of this DCO type as a function of
metallicity, $\eta(Z)$.

In [ ]:
formation_efficiency_per_Z = (
    binaries.groupby('metallicity')['formation_efficiency']
    .sum()
    .reindex(metallicity_grid, fill_value=0.0)
)

plt.figure(figsize=(8, 5))
plt.plot(metallicity_grid, formation_efficiency_per_Z, marker='o')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Metallicity $Z$')
plt.ylabel(r'Formation efficiency $\eta(Z)$ [M$_{\odot}^{-1}$]')
plt.title(f'{dco_type} formation efficiency vs. metallicity ({path.split("/")[-2]})')
plt.tight_layout()
plt.show()

## Comparing multiple DCO types

The same file can be re-used for different DCO types (BBH / BNS / BHNS) by
calling `calculate_formation_efficiency_per_binary` again. Using the same
`random_seed` keeps the IMF-sampling normalisation identical across calls,
so the curves below are directly comparable.

In [ ]:
formation_efficiency_per_Z_by_type = {}

for t in ['BBH', 'BNS', 'BHNS']:
    df, z_grid, _ = calculate_formation_efficiency_per_binary(
        path, dco_type=t, m1_min=m1_min, m1_max=m1_max,
        binary_fraction=binary_fraction, random_seed=42,
    )
    formation_efficiency_per_Z_by_type[t] = (
        df.groupby('metallicity')['formation_efficiency'].sum().reindex(z_grid, fill_value=0.0)
    )

plt.figure(figsize=(8, 5))
for t, fe_per_Z in formation_efficiency_per_Z_by_type.items():
    plt.plot(metallicity_grid, fe_per_Z, marker='o', label=t)

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Metallicity $Z$')
plt.ylabel(r'Formation efficiency $\eta(Z)$ [M$_{\odot}^{-1}$]')
plt.legend()
plt.tight_layout()
plt.show()

## Notes for combining with other (non-legacy) formation-efficiency pipelines

- `extra_dco_columns` lets you pull additional columns straight from the
  `doubleCompactObjects` table alongside the formation efficiency, e.g.
  ```python
  binaries, _, _ = calculate_formation_efficiency_per_binary(
      path, dco_type='BHNS',
      extra_dco_columns=['M1', 'M2', 'tform', 'tc', 'separationDCOFormation', 'eccentricityDCOFormation'],
  )
  ```
- `binaries` is a plain `pandas.DataFrame`, so it can be saved for later use
  (e.g. `binaries.to_hdf('bhns_fiducial_fe.h5', key='binaries')` or
  `binaries.to_parquet(...)`) and combined with formation-efficiency tables
  produced by other (non-legacy) simulations downstream, as long as they
  share a common set of columns (e.g. `metallicity`, `formation_efficiency`,
  plus any of the masses / delay times above).